E20448
Wijethunge P. M. H. S.
================================================================================
  Twin Rotor System — Dynamic Modelling, Simulation & SO(3) PID Control
================================================================================

This script models, simulates, and controls a twin-rotor mechanical system.
The system consists of a horizontal arm free to rotate in both the vertical and
azimuthal directions, with a rotor at each end providing thrust forces.

Three coordinate frames are used:
  • Earth (inertial) frame  : e
  • Intermediate frame      : c  (rotated from e about the vertical axis by θ)
  • Body-fixed frame        : b  (rotated from c about the arm axis by φ)

Index
------
  Task 1 — Coordinate transforms, angular velocity, angular momentum derivation
  Task 2 — Open-loop simulation & 3-D animation (yaw spin + pitch swing)
  Task 3 — Geometric PID attitude controller on SO(3) with figure-8 tracking



#  TASK 1 — TWIN ROTOR DYNAMIC MODEL

"""
COORDINATE TRANSFORMATIONS
Two successive rotations map the inertial Earth frame (e) to the body frame (b).

Step 1 — Earth → Intermediate frame (c)
  A rotation of angle θ about the shared vertical (3-axis) gives:

        c = e · R3(θ)

  where the rotation matrix is:

        R3(θ) = | cos θ  -sin θ  0 |
                | sin θ   cos θ  0 |
                |   0       0    1 |

Step 2 — Intermediate → Body frame (b)
  The body frame shares its 1-axis with the intermediate frame (b1 ∥ c1).
  The b2 axis lies along the rotor arm. A rotation of angle φ about the
  shared 1-axis gives:

        b = c · R1(φ)

  where the rotation matrix is:

        R1(φ) = | 1     0       0    |
                | 0   cos φ  -sin φ  |
                | 0   sin φ   cos φ  |

COMBINED ROTATION MATRIX
─────────────────────────
  Substituting c from Step 1 into Step 2:

        b = e · R3(θ) · R1(φ)   ⟹   R = R3(θ) · R1(φ)

  Carrying out the multiplication yields the full Earth-to-body rotation:

        R = | cos θ    -sin θ cos φ    sin θ sin φ |
            | sin θ     cos θ cos φ   -cos θ sin φ |
            |   0         sin φ           cos φ    |

ANGULAR VELOCITY IN THE BODY FRAME
────────────────────────────────────
  Differentiating R = R3·R1 with the product rule and applying the
  kinematic identity  Ṙ = R·Ω̂  (where Ω̂ is the skew-symmetric angular
  velocity matrix) gives, after simplification:

        Ω = [ φ̇,  θ̇ sin φ,  θ̇ cos φ ]ᵀ   (expressed in the body frame)

ANGULAR MOMENTUM IN THE EARTH FRAME
──────────────────────────────────────
  With diagonal inertia tensor I = diag(I1, I2, I3) aligned with the body
  axes, the body-frame angular momentum is:

        Π = I · Ω = [ I1 φ̇,  I2 θ̇ sin φ,  I3 θ̇ cos φ ]ᵀ

  Rotating back to the Earth frame via π = R·Π yields:

        π1 =  I1 φ̇ cos θ − (I2−I3) θ̇ sin θ sin φ cos φ
        π2 =  I1 φ̇ sin θ + (I2−I3) θ̇ cos θ sin φ cos φ
        π3 =  θ̇ (I2 sin²φ + I3 cos²φ)

CONSTRAINT & CONTROL TORQUES
──────────────────────────────
  A constraint torque T2 prevents roll of the rotor arm; it acts along b2:

        τ_constraint (Earth) = R · [0, T2, 0]ᵀ
                              = [ -T2 sin θ cos φ,
                                   T2 cos θ cos φ,
                                   T2 sin φ       ]

  Control inputs u1, u2 produce thrust at angles α, β relative to b1.
  The resulting body-frame control torque is:

        τ_u = [ u1 cos α − u2 cos β,
                0,
                u1 sin α − u2 sin β ]

EQUATIONS OF MOTION (Newton–Euler)
────────────────────────────────────
  The Newton–Euler principle states:  π̇ = T_external

  After transforming to the body frame (pre-multiplying by Rᵀ):

        IΩ̇ + Ω × IΩ = τ_e + τ_u

  The constraint torque magnitude T2 is found by projecting onto the b2 axis:

        T2 = e2ᵀ · (Ω × IΩ + IΩ̇)


# ─────────────────────────────────────────────────────────────────────────────
#  TASK 2 — OPEN-LOOP SIMULATION & ANIMATION


Two realistic input scenarios are simulated to verify the dynamic model:

  Scenario A — Vertical-axis spin (yaw):
    A constant torque from rotor 2 drives the arm to spin about the
    vertical axis. The torque is reversed halfway to slow the rotation.

  Scenario B — Oscillatory pitch (swing):
    A sinusoidally varying torque from rotor 1 causes the arm to swing
    up and down in the vertical plane, demonstrating pitch dynamics.

State vector:  Y = [θ, φ, θ̇, φ̇]

The equations of motion are integrated numerically using the RK45 method
from SciPy, and a 3-D matplotlib animation visualises the arm and fans.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.integrate import solve_ivp
from IPython.display import HTML, display

# ── System parameters ──────────────────────────────────────────────────────
ARM_HALF_LENGTH = 0.4          # Half-length of rotor arm (m)
INERTIA_1 = 0.02               # Moment of inertia about b1 (arm axis)
INERTIA_2 = 0.08               # Moment of inertia about b2
INERTIA_3 = 0.08               # Moment of inertia about b3

ROTOR1_ANGLE = 0               # Thrust angle α of rotor 1 (rad)
ROTOR2_ANGLE = -np.pi / 2      # Thrust angle β of rotor 2 (rad)

L  = ARM_HALF_LENGTH
I1 = INERTIA_1
I2 = INERTIA_2
I3 = INERTIA_3


# Equations of motion

In [ ]:
def twin_rotor_ode(t, state, input_u1, input_u2):

    """
    Computes state derivatives for the twin-rotor system.

    Parameters
    ----------
    t        : float         — current simulation time
    state    : array (4,)    — [θ, φ, θ̇, φ̇]
    input_u1 : callable      — rotor-1 torque as function of time
    input_u2 : callable      — rotor-2 torque as function of time

    Returns
    -------
    list [θ̇, φ̇, θ̈, φ̈]
    """
    theta, phi, dtheta, dphi = state

    u1 = input_u1(t)
    u2 = input_u2(t)

    # Body-frame control torques
    tau1 = u1 * np.cos(ROTOR1_ANGLE) - u2 * np.cos(ROTOR2_ANGLE)
    tau3 = u1 * np.sin(ROTOR1_ANGLE) - u2 * np.sin(ROTOR2_ANGLE)

    # φ̈ — pitch/roll acceleration
    ddphi = (tau1 - (I3 - I2) * dtheta**2 * np.sin(phi) * np.cos(phi)) / I1

    # θ̈ — yaw acceleration (guarded against gimbal-like singularity)
    if abs(np.cos(phi)) < 1e-4:
        ddtheta = 0.0
    else:
        ddtheta = (tau3 - (I2 - I1 - I3) * dtheta * dphi * np.sin(phi)) \
                  / (I3 * np.cos(phi))

    return [dtheta, dphi, ddtheta, ddphi]


# Rotation matrix builder

In [ ]:
def rotation_matrix(theta, phi):
    """Returns the Earth-to-body rotation matrix R(θ, φ)."""
    ct, st = np.cos(theta), np.sin(theta)
    cp, sp = np.cos(phi),   np.sin(phi)
    return np.array([
        [ ct, -st * cp,  st * sp],
        [ st,  ct * cp, -ct * sp],
        [  0,       sp,       cp]
    ])

#  3-D animation helper

In [ ]:
def build_animation(sol, t_grid, title):
    """
    Generates a 3-D matplotlib animation of the rotor arm.

    Parameters
    ----------
    sol    : OdeSolution  — result from solve_ivp
    t_grid : array        — time points corresponding to sol.y columns
    title  : str          — figure title

    Returns
    -------
    HTML object containing the jshtml animation
    """
    theta_arr = sol.y[0]
    phi_arr   = sol.y[1]

    fig = plt.figure(figsize=(8, 6))
    ax  = fig.add_subplot(111, projection='3d')
    lim = 0.6
    ax.set_xlim([-lim, lim]);  ax.set_ylim([-lim, lim]);  ax.set_zlim([-lim, lim])
    ax.set_xlabel('Earth X (m)');  ax.set_ylabel('Earth Y (m)');  ax.set_zlabel('Earth Z (m)')
    ax.set_title(title)

    # Fixed vertical pillar
    ax.plot([0, 0], [0, 0], [-lim, 0], 'k-', lw=4)

    # Animated elements
    arm_line,  = ax.plot([], [], [], 'k-', lw=5)
    fan1_line, = ax.plot([], [], [], 'r-', lw=2)
    fan2_line, = ax.plot([], [], [], 'b-', lw=2)

    # Fan disc geometry in body frame
    angles      = np.linspace(0, 2 * np.pi, 20)
    fan_radius  = 0.1
    cc          = fan_radius * np.cos(angles)
    cs          = fan_radius * np.sin(angles)

    alpha, beta = ROTOR1_ANGLE, ROTOR2_ANGLE

    fan1_body = np.array([
        -np.sin(alpha) * cc,
         cs + L,
         np.cos(alpha) * cc
    ])
    fan2_body = np.array([
        -np.sin(beta) * cc,
         cs - L,
         np.cos(beta) * cc
    ])

    def update_frame(k):
        R = rotation_matrix(theta_arr[k], phi_arr[k])

        # Arm endpoints (along b2)
        tip_pos  = R @ np.array([0,  L, 0])
        tail_pos = R @ np.array([0, -L, 0])
        arm_line.set_data(  [tail_pos[0], tip_pos[0]], [tail_pos[1], tip_pos[1]])
        arm_line.set_3d_properties([tail_pos[2], tip_pos[2]])

        # Fan discs
        f1 = R @ fan1_body
        fan1_line.set_data(f1[0], f1[1]);  fan1_line.set_3d_properties(f1[2])

        f2 = R @ fan2_body
        fan2_line.set_data(f2[0], f2[1]);  fan2_line.set_3d_properties(f2[2])

        return arm_line, fan1_line, fan2_line

    ani = animation.FuncAnimation(
        fig, update_frame, frames=len(t_grid), interval=40, blit=True
    )
    plt.close(fig)
    return HTML(ani.to_jshtml())

#Simulation settings
TIME_SPAN  = (0, 8)
TIME_GRID  = np.linspace(TIME_SPAN[0], TIME_SPAN[1], 200)
INIT_STATE = [0.0, 0.0, 0.0, 0.0]   # Start at rest, arm horizontal

# Scenario A — Yaw (vertical-axis spin)

In [ ]:
def yaw_u1(t): return 0.0
def yaw_u2(t): return 0.05 if t < 4 else -0.05

sol_yaw = solve_ivp(
    twin_rotor_ode, TIME_SPAN, INIT_STATE,
    t_eval=TIME_GRID, args=(yaw_u1, yaw_u2), method='RK45'
)

print("Generating Scenario A animation …")
display(build_animation(sol_yaw, TIME_GRID, "Scenario A: Vertical Axis Spin (Yaw)"))


# ── Scenario B — Pitch (oscillatory vertical swing) ───────────────────────
def pitch_u1(t): return 0.02 * np.sin(np.pi * t)
def pitch_u2(t): return 0.0

sol_pitch = solve_ivp(
    twin_rotor_ode, TIME_SPAN, INIT_STATE,
    t_eval=TIME_GRID, args=(pitch_u1, pitch_u2), method='RK45'
)

print("Generating Scenario B animation …")
display(build_animation(sol_pitch, TIME_GRID, "Scenario B: Oscillatory Pitch Swing"))

#  TASK 3 — GEOMETRIC PID ATTITUDE CONTROLLER ON SO(3)

MOTIVATION FOR WORKING DIRECTLY ON SO(3)

  Euler-angle representations (e.g. pitch–yaw) develop coordinate singularities
  when the pitch angle approaches ±90°, a phenomenon known as gimbal lock.
  To build a controller that remains well-defined for all orientations, attitude
  is represented directly as a rotation matrix R ∈ SO(3).

CONFIGURATION ERROR

  Let R  be the actual rotation (body→Earth) and Rr the desired rotation.
  The attitude error matrix is defined as:

        Re = Rr · Rᵀ

  When the system is perfectly aligned with the reference (R = Rr), Re = I.
  The control objective is therefore:

        Re(t) → I   as   t → ∞

ERROR KINEMATICS

  Differentiating Re and using  Ṙ = ω̂R  gives:

        Ṙe = ω̂e · Re   where   ω̂e = ω̂r − R̂eω

  In vector form the angular-velocity error is:

        ωe = ωr − Re · ω

ANGULAR MOMENTUM ERROR

  Define the spatial reference momentum  πr = R · Πr  and the actual
  spatial momentum  π = R · IΩ.  The momentum error is:

        πe = πr − π

P–I–D CONTROL COMPONENTS ON SO(3)

  The attitude error vector is extracted from Re via the vee operator:

        eR = vee( (Re − Reᵀ) / 2 )

  Proportional term:   τP = Kp · eR
  Derivative term:     τD = Kd · πe
  Integral term:       τI = Ki · eI,   where  ėI = eR

  Total feedback torque:
        τ_feedback = τP + τD + τI

  Combined with a feedforward term to cancel known reference dynamics, this
  gives the required spatial torque Treq, which is then mapped back to the
  body frame to yield the actual actuator commands u1, u2.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.integrate import solve_ivp
from IPython.display import HTML, display

# ── System parameters (same as Task 2) ────────────────────────────────────
L  = 0.4
I1, I2, I3 = 0.02, 0.08, 0.08
II = np.diag([I1, I2, I3])

ROTOR1_ANGLE = 0
ROTOR2_ANGLE = -np.pi / 2

# ── Controller gain matrices ───────────────────────────────────────────────
Kp = np.diag([2.0, 2.0, 2.0])   # Proportional — attitude error
Kd = np.diag([1.0, 1.0, 1.0])   # Derivative   — momentum error
Ki = np.diag([0.5, 0.5, 0.5])   # Integral     — accumulated attitude error


# ── SO(3) utility functions ────────────────────────────────────────────────
def get_R(theta, phi):
    """Builds the rotation matrix from (θ, φ)."""
    ct, st = np.cos(theta), np.sin(theta)
    cp, sp = np.cos(phi),   np.sin(phi)
    return np.array([
        [ ct, -st * cp,  st * sp],
        [ st,  ct * cp, -ct * sp],
        [  0,       sp,       cp]
    ])

In [ ]:
def vee(M):
    """Extracts the axial vector from a skew-symmetric 3×3 matrix."""
    return np.array([M[2, 1], M[0, 2], M[1, 0]])


def skew(v):
    """Constructs the 3×3 skew-symmetric (cross-product) matrix for vector v."""
    return np.array([
        [    0, -v[2],  v[1]],
        [ v[2],     0, -v[0]],
        [-v[1],  v[0],     0]
    ])


In [ ]:

# ── Reference trajectory — figure-8 in angle space
def reference_trajectory(t):
    """
    Generates the desired rotation, angular momentum, and its time
    derivative at time t.

    The reference traces a smooth figure-8 pattern:
        θr(t) = 0.8 sin(0.5t)
        φr(t) = 0.4 cos(1.0t)
    """
    theta_r  =  0.8 * np.sin(0.5 * t)
    phi_r    =  0.4 * np.cos(1.0 * t)
    dtheta_r =  0.4 * np.cos(0.5 * t)    # θ̇r
    dphi_r   = -0.4 * np.sin(1.0 * t)    # φ̇r

    Rr      = get_R(theta_r, phi_r)
    omega_r = np.array([dphi_r,
                        dtheta_r * np.sin(phi_r),
                        dtheta_r * np.cos(phi_r)])
    Pi_r    = II @ omega_r
    dPi_r   = np.zeros(3)               # Feedforward simplified to zero

    return Rr, Pi_r, dPi_r

In [ ]:
# ── Closed-loop dynamics
def pid_closed_loop(t, state):
    """
    Full state derivative under the geometric PID controller.

    State layout: [θ, φ, θ̇, φ̇, eI_x, eI_y, eI_z]
      where (eI_x, eI_y, eI_z) is the integral of the attitude error vector.
    """
    theta, phi, dtheta, dphi = state[:4]
    eI = state[4:7]

    # ── Current kinematics
    R       = get_R(theta, phi)
    omega_b = np.array([dphi,
                        dtheta * np.sin(phi),
                        dtheta * np.cos(phi)])     # Body-frame angular velocity
    omega_s = R @ omega_b                          # Spatial (Earth) frame
    Pi      = II @ omega_b                         # Body-frame angular momentum
    pi_s    = R @ Pi                               # Spatial angular momentum

    # ── Reference state
    Rr, Pi_r, dPi_r = reference_trajectory(t)
    pi_r = R @ Pi_r       # Reference momentum mapped through actual R

    # ── Errors
    Re  = Rr @ R.T                                 # Attitude error matrix
    eR  = vee((Re - Re.T) / 2.0)                  # Attitude error vector
    pi_e = pi_r - pi_s                             # Momentum error

    # ── PID control law (spatial frame)
    feedforward = R @ dPi_r + np.cross(omega_s, pi_r)
    feedback    = Kp @ eR + Kd @ pi_e + Ki @ eI
    T_req       = feedforward + feedback

    # ── Map required spatial torque back to body-frame actuator commands ───
    tau_body = R.T @ T_req
    u1 = tau_body[0]   # Drives φ̈ (pitch)
    u2 = tau_body[2]   # Drives θ̈ (yaw)

    # ── Plant equations of motion
    ddphi = (u1 - (I3 - I2) * dtheta**2 * np.sin(phi) * np.cos(phi)) / I1

    if abs(np.cos(phi)) < 1e-4:
        ddtheta = 0.0
    else:
        ddtheta = (u2 - (I2 - I1 - I3) * dtheta * dphi * np.sin(phi)) \
                  / (I3 * np.cos(phi))

    return [dtheta, dphi, ddtheta, ddphi, eR[0], eR[1], eR[2]]


# ── Simulate
T_SPAN = (0, 15)
T_GRID = np.linspace(T_SPAN[0], T_SPAN[1], 300)
INIT_7 = [0.0] * 7    # Start perfectly at rest, arm horizontal

print("Simulating geometric PID tracking controller …")
sol_pid = solve_ivp(pid_closed_loop, T_SPAN, INIT_7,
                    t_eval=T_GRID, method='RK45')



In [ ]:
# ── Animate with side-by-side performance plot
def build_pid_animation(sol, t_grid):
    """
    Creates a two-panel animation:
      Left  — 3-D view of actual arm (black) and reference arm (green dashed)
      Right — Time histories of actual vs reference θ and φ
    """
    theta_arr = sol.y[0]
    phi_arr   = sol.y[1]

    # Reference trajectories for comparison
    theta_ref = 0.8 * np.sin(0.5 * t_grid)
    phi_ref   = 0.4 * np.cos(1.0 * t_grid)

    fig = plt.figure(figsize=(12, 5))

    # ── 3-D panel
    ax3d = fig.add_subplot(121, projection='3d')
    lim  = 0.6
    ax3d.set_xlim([-lim, lim]); ax3d.set_ylim([-lim, lim]); ax3d.set_zlim([-lim, lim])
    ax3d.set_xlabel('X (m)'); ax3d.set_ylabel('Y (m)'); ax3d.set_zlabel('Z (m)')
    ax3d.set_title('PID Tracking — 3D View')
    ax3d.plot([0, 0], [0, 0], [-lim, 0], 'k-', lw=3)   # pillar

    arm_act, = ax3d.plot([], [], [], 'k-',  lw=4, label='Actual')
    arm_ref, = ax3d.plot([], [], [], 'g--', lw=2, label='Reference')
    fan1_act, = ax3d.plot([], [], [], 'r-', lw=2)
    fan2_act, = ax3d.plot([], [], [], 'b-', lw=2)
    ax3d.legend(loc='upper right', fontsize=7)

    angles     = np.linspace(0, 2 * np.pi, 20)
    fan_r      = 0.1
    cc, cs     = fan_r * np.cos(angles), fan_r * np.sin(angles)
    alpha, beta = ROTOR1_ANGLE, ROTOR2_ANGLE
    fan1_body  = np.array([-np.sin(alpha)*cc, cs+L, np.cos(alpha)*cc])
    fan2_body  = np.array([-np.sin(beta) *cc, cs-L, np.cos(beta) *cc])

    # ── Performance panel
    ax2d = fig.add_subplot(122)
    ax2d.set_xlim(t_grid[0], t_grid[-1])
    ax2d.set_ylim(-1.2, 1.2)
    ax2d.set_xlabel('Time (s)'); ax2d.set_ylabel('Angle (rad)')
    ax2d.set_title('Angle Tracking Performance')
    ax2d.plot(t_grid, theta_ref, 'g--', lw=1, label='θ ref')
    ax2d.plot(t_grid, phi_ref,   'b--', lw=1, label='φ ref')
    theta_line, = ax2d.plot([], [], 'g-', lw=2, label='θ actual')
    phi_line,   = ax2d.plot([], [], 'b-', lw=2, label='φ actual')
    time_cursor, = ax2d.plot([], [], 'k|', ms=10)
    ax2d.legend(fontsize=7)

    plt.tight_layout()

    def update(k):
        # Actual arm
        R = get_R(theta_arr[k], phi_arr[k])
        p1 = R @ np.array([0,  L, 0])
        p2 = R @ np.array([0, -L, 0])
        arm_act.set_data(   [p2[0], p1[0]], [p2[1], p1[1]])
        arm_act.set_3d_properties([p2[2], p1[2]])

        # Reference arm
        Rr = get_R(theta_ref[k], phi_ref[k])
        r1 = Rr @ np.array([0,  L, 0])
        r2 = Rr @ np.array([0, -L, 0])
        arm_ref.set_data(   [r2[0], r1[0]], [r2[1], r1[1]])
        arm_ref.set_3d_properties([r2[2], r1[2]])

        # Fans
        f1 = R @ fan1_body
        fan1_act.set_data(f1[0], f1[1]); fan1_act.set_3d_properties(f1[2])
        f2 = R @ fan2_body
        fan2_act.set_data(f2[0], f2[1]); fan2_act.set_3d_properties(f2[2])

        # Time-series plots (growing traces)
        theta_line.set_data(t_grid[:k], theta_arr[:k])
        phi_line.set_data(  t_grid[:k], phi_arr[:k])
        time_cursor.set_data([t_grid[k]], [0])

        return arm_act, arm_ref, fan1_act, fan2_act, theta_line, phi_line, time_cursor

    ani = animation.FuncAnimation(
        fig, update, frames=len(t_grid), interval=50, blit=True
    )
    plt.close(fig)
    return HTML(ani.to_jshtml())


print("Generating PID controller animation …")
display(build_pid_animation(sol_pid, T_GRID))